# Advanced Preprocessing (Fixed Emoji & Font Normalization)
Tahap ini bertujuan untuk menormalisasi teks komentar YouTube. Strategi mencakup:
1. Emoji Conversion (Demojize -> Text) **[URUTAN PERTAMA]**
2. Unicode Normalization (Bold/Italic -> Normal)
3. URL Masking ([URL])
4. Indonesian Slang Normalization (saka-nlp)
5. Character Repetition Reduction

In [20]:
import pandas as pd
import re
import unicodedata
import saka
import emoji
from tqdm import tqdm

# Setup tqdm for pandas
tqdm.pandas()

df = pd.read_csv('../Dataset/dataset_prepared.csv')
print(f"Total data: {len(df)}")

Total data: 70404


## Definisikan Pipeline Preprocessing Lanjutan (Fixed)

In [21]:
def final_preprocessing_pipeline(text):
    if not isinstance(text, str): return ""
    
    # 1. Emoji to Text (Demojize) - Harus paling awal agar tidak terhapus proses ASCII
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 2. Unicode Normalization (Bold/Italic -> Normal)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    
    # 3. URL Masking
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text)
    text = re.sub(r'bit\s?[\.\,]\s?ly/\w+', '[URL]', text)
    text = re.sub(r'\w+\s?\[dot\]\s?\w+', '[URL]', text)
    
    # 4. Lowercase
    text = text.lower()
    
    # 5. Symbol & Cleaning (Keep alphanumeric, [URL], and underscore/colon from demojize)
    text = re.sub(r'[^a-zA-Z0-9\s\[\]\_:]', ' ', text)
    
    # 6. Normalize with saka-nlp (Indonesian slang)
    text = saka.normalize(text)
    
    # 7. Reduce repeated chars
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 8. Final Whitespace Cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Applying pipeline... (This may take a few minutes)")
df['text_clean'] = df['text'].progress_apply(final_preprocessing_pipeline)
print("Preprocessing selesai.")

Applying pipeline... (This may take a few minutes)


100%|██████████| 70404/70404 [00:03<00:00, 19981.07it/s]

Preprocessing selesai.


## Cek Hasil Perbandingan

In [22]:
df[['text', 'text_clean', 'label']].head(20)

,text,text_clean,label
0,makin yakin abis baca review lain tentang ✌✌𝐒𝐆...,makin yakin habis baca review lain tentang vic...,1
1,paling suka model h2 😍🔥,paling suka model h2 smiling_face_with_heart e...,0
2,mobilnya udah hancur 🥺,mobilnya sudah hancur pleading_face,0
3,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin aku jadi sultan,manut88benar2 bikin aku jadi sultan,1
4,semoga lekas recover mobilnya mas dipo,semoga lekas recover mobilnya mas dipo,0
5,pantes tongkrongan maen alexis17,pantes tongkrongan main alexis17,1
6,gua udah ga bisa bayangin hidup tanpa alexis-1...,saya sudah ga bisa bayangin hidup tanpa alexis...,1
7,langsung percaya lihat hasilnya,langsung percaya lihat hasilnya,1
8,orang bilang alexis17 tuh miracle banget !,orang bilang alexis17 tuh miracle banget,1
9,yapping mulu anjir kagak ada jawaban sistemati...,yapping melulu anjing tidak ada jawaban sistem...,0


## Simpan Dataset Final

In [23]:
# Hapus baris yang teksnya kosong setelah dibersihkan
df = df[df['text_clean'] != ""]

# Gunakan text_clean sebagai kolom text utama
df_final = df[['text_clean', 'label']].rename(columns={'text_clean': 'text'})

df_final.to_csv('../Dataset/dataset_clean_final.csv', index=False)
print(f"Dataset final disimpan. Total baris: {len(df_final)}")

Dataset final disimpan. Total baris: 70379
